# Week 3 Step 2: Give the LLM the schema

Gemini has never seen my database. Before I ask it to write SQL, I have to hand it the exact table names, column names, and joins.

I keep that text in `src/llm/schema.py`: a compact schema string, a few business rules it cannot guess (churn lives in `churn_labels`, revenue is successful payments only), and five example question-to-SQL pairs. The system prompt also says: return **one SELECT only**, no markdown, and `CANNOT_ANSWER` if the question is outside this schema.

I am not running the SQL yet. That is the validation layer in Step 3.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.client import ask_llm
from src.llm.schema import (
    SQL_EXAMPLES,
    inspect_live_schema,
    schema_text,
    text_to_sql_system_prompt,
)

print(schema_text())

customers(customer_id PK, first_name, last_name, age, gender, country, state_or_region, city, registration_date, acquisition_channel, customer_segment, preferred_language)
subscription_plans(plan_id PK, plan_name, monthly_price, video_quality, max_devices, advertisements)
content(content_id PK, title, content_type, genre, release_year, duration_minutes, content_language, production_country, maturity_rating)
subscriptions(subscription_id PK, customer_id FK->customers, plan_id FK->subscription_plans, subscription_start_date, subscription_end_date, subscription_status, auto_renew, cancellation_date, cancellation_reason, monthly_price)
viewing_activity(viewing_id PK, customer_id FK->customers, content_id FK->content, viewing_date, watch_duration_minutes, completion_percentage, device_type, login_location, session_duration_minutes)
payments(payment_id PK, customer_id FK->customers, subscription_id FK->subscriptions, payment_date, amount, payment_method, payment_status, failed_payment_reason

Week 1's `sql_business_questions.sql` file was empty, so these five examples use the real column names from `schema.sql` instead of copied Week 1 answers.

In [2]:
for i, ex in enumerate(SQL_EXAMPLES, start=1):
    print(f"Example {i}: {ex['question']}")
    print(ex["sql"])
    print()

Example 1: How many customers are in each country?
SELECT country, COUNT(*) AS n_customers
FROM customers
GROUP BY country
ORDER BY n_customers DESC

Example 2: What is the churn rate by country?
SELECT c.country,
       AVG(CASE WHEN cl.churned THEN 1.0 ELSE 0.0 END) AS churn_rate
FROM customers c
JOIN churn_labels cl ON cl.customer_id = c.customer_id
GROUP BY c.country
ORDER BY churn_rate DESC

Example 3: What is total successful payment revenue by plan name?
SELECT sp.plan_name, SUM(p.amount) AS revenue
FROM payments p
JOIN subscriptions s ON s.subscription_id = p.subscription_id
JOIN subscription_plans sp ON sp.plan_id = s.plan_id
WHERE LOWER(p.payment_status) = 'success'
GROUP BY sp.plan_name
ORDER BY revenue DESC

Example 4: What is average customer satisfaction by issue category?
SELECT issue_category, AVG(customer_satisfaction_score) AS avg_csat
FROM support_tickets
WHERE customer_satisfaction_score IS NOT NULL
GROUP BY issue_category
ORDER BY avg_csat

Example 5: Which 10 titl

In [3]:
system_prompt = text_to_sql_system_prompt()
print(system_prompt)

You write PostgreSQL for the Netflix Customer Intelligence database.

Return only one PostgreSQL SELECT query.
Do not add an explanation, markdown fences, or comments.
Use only the tables and columns listed below.
If the question cannot be answered from this schema, return the single word CANNOT_ANSWER.

Schema:
customers(customer_id PK, first_name, last_name, age, gender, country, state_or_region, city, registration_date, acquisition_channel, customer_segment, preferred_language)
subscription_plans(plan_id PK, plan_name, monthly_price, video_quality, max_devices, advertisements)
content(content_id PK, title, content_type, genre, release_year, duration_minutes, content_language, production_country, maturity_rating)
subscriptions(subscription_id PK, customer_id FK->customers, plan_id FK->subscription_plans, subscription_start_date, subscription_end_date, subscription_status, auto_renew, cancellation_date, cancellation_reason, monthly_price)
viewing_activity(viewing_id PK, customer_id FK

In [4]:
import importlib
import os
import re

import src.llm.schema as schema_mod

importlib.reload(schema_mod)
from src.llm.schema import TABLES, inspect_live_schema

url = os.getenv("DATABASE_URL") or ""
print("trying:", re.sub(r":([^:@/]+)@", ":***@", url) if url else "no DATABASE_URL")

live = inspect_live_schema()
if live is None:
    print("Postgres is not reachable. I am using schema.sql names for the prompt.")
    print("reason:", schema_mod.LAST_INSPECT_ERROR)
else:
    print("tables in Postgres:", sorted(live))
    for name in TABLES:
        if name not in live:
            print("missing in db:", name)
        elif set(live[name]) != set(TABLES[name]["columns"]):
            print(name, "column mismatch")
            print("  db ", sorted(live[name]))
            print("  file", TABLES[name]["columns"])
    print("live inspect finished")

Postgres is not reachable. I am using schema.sql names for the prompt.
reason: None
Start Docker Desktop, then run: docker start netflix-db


I now hand the prompt to Gemini on a question it should be able to answer, and on one it should refuse.

In [5]:
question = "How many customers churned in each country?"
sql = ask_llm(system_prompt, question, max_tokens=400)
print("question:", question)
print(sql)

print()
refused = ask_llm(
    system_prompt,
    "What is the CEO's email address?",
    max_tokens=50,
)
print("out of schema:")
print(refused)

question: How many customers churned in each country?
SELECT c.country, COUNT(c.customer_id) AS churned_customers_count
FROM customers c
JOIN churn_labels cl ON c.customer_id = cl.customer_id
WHERE cl.churned = TRUE
GROUP BY c.country
ORDER BY churned_customers_count DESC

out of schema:

